# Aero objective — does the blade make wind?

**What this notebook is.** The single most important thing about a fan is that it *moves
air*. This notebook runs the **aero objective** for the redesigned solid blade: it takes a
blade's shape, simulates the airflow with a real CFD solver (SU2), and reports **`J_fan`**
— the net amount of air the blade pushes per stroke. This is the quantity the optimizer
will eventually *maximize*; everything else (weight, folding, stiffness) is a constraint
that protects it.

**Why "aero-first".** The blade's shape is what makes wind; the internal structure just
supports that shape. So we measure wind *first*, before optimizing anything structural.

**The pipeline** (all the real logic lives in `src/fanopt/`, tested — this notebook only
orchestrates and visualizes):

```
BladeParams ──▶ blade_slice_polygons ──▶ cascade mesh ──▶ SU2 (steady + unsteady) ──▶ J_fan
   (shape)         (2D cross-section)      (the grid)      (the airflow sim)      (net wind)
```

We'll run three shapes — **flat**, **cambered**, **zigzag** — and see which pushes the
most air, and confirm that a *flat* blade pushes ~zero net air (the "parachute" sanity
check: it shoves air equally on the up- and down-stroke, so they cancel).

## 1. Setup

All imports live in this one cell (project rule: imports at the top). We also locate the
SU2 solver. `find_su2()` checks the `$SU2_RUN` environment variable and your `PATH`; set
`SU2_BIN` below if yours lives elsewhere. SU2 runs locally on the M3 (or on Colab with the
SU2 build on `PATH`).

In [ ]:
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from fanopt.geometry.blade import BladeParams
from fanopt.cfd.blade_slice import blade_slice_polygons
from fanopt.cfd.blade_aero import evaluate_blade_aero
from fanopt.cfd.phase3 import find_su2
from fanopt.cfd.parsers import parse_su2_unsteady_force_series

# If SU2 isn't on PATH / $SU2_RUN, point SU2_BIN at your SU2_CFD binary:
SU2_BIN = os.environ.get("SU2_CFD_BIN") or find_su2()
WORKROOT = Path("aero_objective_runs")   # per-design SU2 working dirs land here
print("SU2:", SU2_BIN or "NOT FOUND — set SU2_CFD_BIN to your SU2_CFD path")

## 2. Three blade shapes

Each blade's aero surface is a **displacement grid** — a small grid of surface offsets the
optimizer controls. We build three hand-made examples to compare:

- **flat** — no displacement (the parachute baseline)
- **cambered** — a smooth arch across the chord (classic lift-generating shape)
- **zigzag** — a corrugation (a louver-like shape that trades a different way)

The plot below shows each blade's **cross-section** — exactly the 2D geometry the CFD
meshes. The chord runs tangentially (rib to rib); the airflow (not drawn) comes from below
in the *z* direction, so the blade sits broadside to the air it pushes.

In [ ]:
def make_blade(grid):
    return BladeParams(blade_count=10, rib_bow_mid_m=0.010, rib_bow_tip_m=0.020,
                       t_rib_hub_m=0.0025, t_rib_tip_m=0.0035,
                       panel_offsets_m=grid, panel_thickness_nom_m=0.0016)

SHAPES = {
    "flat":     tuple((0.0, 0.0, 0.0) for _ in range(4)),
    "cambered": ((0.0004,0.0009,0.0004),(0.0005,0.0011,0.0005),
                 (0.0006,0.0013,0.0006),(0.0007,0.0015,0.0007)),
    "zigzag":   ((0.0009,-0.0009,0.0009),(0.0011,-0.0011,0.0011),
                 (0.0013,-0.0013,0.0013),(0.0015,-0.0015,0.0015)),
}
blades = {name: make_blade(g) for name, g in SHAPES.items()}

fig, ax = plt.subplots(figsize=(11, 3))
for name, blade in blades.items():
    poly = blade_slice_polygons(blade, radial_u=0.6, n_panels=1, n_samples=40)[0]
    ax.plot(poly[:, 1]*1000, poly[:, 0]*1000, lw=1.8, label=name)
ax.set_xlabel("tangential / chord (mm)"); ax.set_ylabel("z (mm)")
ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.legend(); ax.set_title("Blade cross-sections")
plt.show()

## 3. The aero model (what `J_fan` means)

We model the fan stroke as a **plunging cascade**: a row of blade cross-sections strokes
up and down through the air (the waving motion), and SU2 solves the resulting unsteady
flow. From the force history we reduce two numbers:

- **`J_fan`** — the **net momentum flux** averaged over a full plunge cycle. A symmetric
  (flat) blade pushes air down on the down-stroke and up on the up-stroke, so its net is
  ~0. A shaped blade breaks that symmetry → a non-zero net → **directional wind**. This is
  the objective.
- **`steady_CD`** — a cheap steady-flow drag number, used as a fast screening proxy.

**Two things to keep in mind when reading the numbers:**
1. The unsteady run uses `MACH = 1e-9` (a locked low-speed convention), which **inflates
   the absolute force scale** enormously (~10¹⁰–10¹³). So **only *relative* comparisons
   between shapes are meaningful** — never the absolute value.
2. `J_fan` is the small *net* left over after a huge up/down oscillation cancels. We'll
   plot the raw force series so you can *see* the oscillation and the net.

## 4. Run the CFD  ⏱️ (slow — a few minutes per shape)

This cell runs SU2 twice per blade (a steady solve + an unsteady plunging solve). With the
light settings below it's ~3–5 min per shape. Results are cached per-design in `WORKROOT`;
re-running re-uses the finished working dirs. **Production runs want `n_cycles >= 5`** for
a converged `J_fan` (here we use 3 to keep it quick).

In [ ]:
assert SU2_BIN, "SU2 not found — set SU2_CFD_BIN"
results = {}
for name, blade in blades.items():
    print(f"[eval] {name} ...", flush=True)
    results[name] = evaluate_blade_aero(
        blade, WORKROOT / name, su2_bin=SU2_BIN,
        radial_u=0.6, n_panels=3, n_samples=28,
        n_cycles=3, inner_iter=40, steps_per_cycle=100,
    )
    r = results[name]
    print(f"    J_fan = {r.j_fan:.3e}   steady_CD = {r.steady_cd:.3e}   nodes = {r.n_nodes}")

## 5. Results

**Left:** `J_fan` per shape (relative — the scale is the inflated `MACH=1e-9` scale).
**Right:** the raw plunging force over the last two cycles. Notice the force swings ±10¹³
(the stroke), while the *net* (dashed mean) is a thin sliver near zero for **flat** (the
parachute) and lifts off zero for **cambered** / **zigzag** (directional wind). That thin
sliver *is* `J_fan`.

In [ ]:
names = list(results)
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.2))

axL.bar(names, [results[n].j_fan for n in names], color="#4477aa")
axL.set_title("J_fan by shape (relative)"); axL.set_ylabel("net momentum flux (a.u.)")
axL.grid(alpha=0.3, axis="y")

for name in names:
    s = parse_su2_unsteady_force_series(WORKROOT / name / "history.csv")
    spc = s.size // 3
    tail = s[spc:]                      # drop cycle-0 startup transient
    axR.plot(tail, lw=1.0, label=name)
    axR.axhline(tail.mean(), ls="--", lw=0.9, alpha=0.7)
axR.set_title("Plunging force (last 2 cycles) + net mean (dashed)")
axR.set_xlabel("time step"); axR.set_ylabel("force (a.u.)"); axR.legend(); axR.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Reading it + what's next

**What we confirmed:**
- The full aero objective runs end to end: a blade's *shape* → a wind number.
- **flat ≈ 0 net** (parachute), while **cambered / zigzag** produce a real net → the shape
  (the displacement grid) is a genuine, dominant aero lever. The optimizer has signal.

**Caveats (be honest about these):**
- Absolute `J_fan` is on the inflated `MACH=1e-9` scale → **relative only**.
- 3 cycles ≈ 2 usable → **not fully converged** (~30% cycle-to-cycle). Bump `n_cycles` to
  5+ for a stable number before trusting a single design's value.
- This is a single mid-radius 2D slice at screening fidelity — not the full 3D blade.

**Next:** wrap `evaluate_blade_aero` as the BO's primary objective (maximize `J_fan`) with
mass / fold / deflection as constraints, then run the multi-objective campaign.